In [1]:
import sys
from pathlib import Path

sys.path.append(
    str(Path.cwd().parent / "src")
)
from extraction.gee import initialize

initialize()

✓ Earth Engine initialized successfully.


In [2]:
import ee

collection = ee.ImageCollection(
    "projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a"
)
print(collection.first().getInfo())

{'type': 'Image', 'bands': [{'id': 'GPP', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}, {'id': 'ET', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 4294967295}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}, {'id': 'Ec', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}, {'id': 'Es', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}, {'id': 'Ei', 'data_type': {'type': 'PixelType', 'precision': 'int', 'm

In [3]:
image = collection.first()

print(image.select("ET").reduceRegion(reducer=ee.Reducer.first(), geometry=ee.Geometry.Point([77.5,21.8]), scale=500, maxPixels=1e9,).getInfo())

{'ET': 140}


In [4]:
print(collection.size().getInfo())
print(collection.first().date().format().getInfo())
print(collection.sort("system:time_start", False).first().date().format().getInfo())

1142
2000-03-05T00:00:00
2024-12-26T00:00:00


In [5]:
dates = collection.aggregate_array("system:time_start")
print(ee.List(dates).slice(0,5).map(lambda x: ee.Date(x).format("YYYY-MM-dd")).getInfo())

['2000-03-05', '2000-03-13', '2000-03-21', '2000-03-29', '2000-04-06']


In [6]:
region = ee.Geometry.Point([77.5,21.8]).buffer(700)
print(image.select("ET").reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=500, maxPixels=1e9).getInfo())

{'ET': 92.11976047904193}


In [7]:
print(image.getInfo()["bands"][1])
print(image.select("ET").getInfo())

{'id': 'ET', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 4294967295}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}
{'type': 'Image', 'bands': [{'id': 'ET', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 4294967295}, 'dimensions': [86400, 35760], 'crs': 'EPSG:4326', 'crs_transform': [0.004166666666666667, 0, -180, 0, -0.004166666666666667, 89]}], 'version': 1768059128223670.0, 'id': 'projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a/2000-03-05', 'properties': {'system:time_start': 952214400000, 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'system:asset_size': 3251054722, 'system:index': '2000-03-05'}}


In [8]:
from extraction.sites import get_site
from extraction.products import get_product
from extraction.extractor import extract_timeseries

site = get_site("BFT")
product = get_product("PMLV2")
df = extract_timeseries(
    site,
    product,
    "2016-01-01",
    "2016-12-31",
)

df.head()

,Date,DoY,ET
0,2016-01-01,1,0.860227
1,2016-01-09,9,1.048004
2,2016-01-17,17,0.936115
3,2016-01-25,25,1.055499
4,2016-02-02,33,1.041124
